# v8 RELATED languages, continual (GPU 0): Hindi → Gujarati → Punjabi → Hindi → ...
A new model from scratch, same recipe as the distant-language continual run (`v4b_continual_s0`, Hindi → Chinese → Arabic).
The only intended difference: the three languages are closely related. As in the distant run, each has its own script (Devanagari, Gujarati, Gurmukhi) and output is restricted to that script.
* Data: **balanced**, exactly 993,016 training pairs per language (Hindi = IITB, Gujarati and Punjabi = Samanantar), same cleaning as the Hindi data.
* Tokenizer: `tokenizer_v8`, the same recipe as tokenizer_v4, on English + Hindi + Gujarati + Punjabi; tags `<2hi>` `<2gu>` `<2pa>`.
* New names only (`common_v8`, `tokenizer_v8`, `data/en-gu`, `data/en-pa`, run `v8_indic_continual_s0`): nothing existing is touched.

**Just Run All.** The first cell prepares data + tokenizer (about 20-30 min the first time; it stops with an error if a language has too little data). Rerunning resumes from the last saved phase.

In [1]:
import os, sys, subprocess
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CML_LANGSET"] = "indic"             # Hindi, Gujarati, Punjabi + tokenizer_v8 (must be set before importing common_v8)
os.environ["HF_HUB_DISABLE_XET"] = "1"
REPO = "/workspace/shreshth"; os.chdir(REPO); sys.path.insert(0, REPO)
subprocess.run([sys.executable, "-u", "prep_indic_data.py"], check=True)

Hindi training pairs: 993,016 -> Gujarati and Punjabi get exactly the same number


Generating train split: 100%|██████████| 3067790/3067790 [00:02<00:00, 1313570.22 examples/s]


gu: 3,067,790 raw -> 3,002,614 clean pairs
wrote data/en-gu/val.jsonl 2,000
wrote data/en-gu/train.jsonl 993,016


Generating train split: 100%|██████████| 2980383/2980383 [00:02<00:00, 1112238.13 examples/s]


pa: 2,980,383 raw -> 2,851,669 clean pairs
wrote data/en-pa/val.jsonl 2,000
wrote data/en-pa/train.jsonl 993,016


Generating dev split: 997 examples [00:00, 179889.92 examples/s]
Generating devtest split: 1012 examples [00:00, 189568.83 examples/s]


... (2 outputs left out)

wrote data/flores/dev.pa.jsonl 997
wrote data/flores/devtest.pa.jsonl 1012
tokenizer text en: 160,979 lines, 20,000,313 characters
tokenizer text hi: 168,705 lines, 20,000,181 characters
tokenizer text gu: 230,540 lines, 20,000,475 characters
tokenizer text pa: 198,687 lines, 20,000,102 characters


I0000 00:00:1793690930.711182    7181 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: data/tok_train_v8.txt
  input_format: 
  model_prefix: tokenizer_v8/spm
  model_type: UNIGRAM
  vocab_size: 32000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 4000000
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4096
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: <2hi>
  user_defined_symbols: <2gu>
  user_defined_symbols: <2pa>
  required_chars: 
  byte_fallback: 1
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: 2
  eos_id: 3
  

  hi: 1.37 pieces per word (2,000 validation sentences)
  gu: 1.52 pieces per word (2,000 validation sentences)
  pa: 1.31 pieces per word (2,000 validation sentences)
data ready: {'hi': 993016, 'gu': 993016, 'pa': 993016}


CompletedProcess(args=['/venv/main/bin/python', '-u', 'prep_indic_data.py'], returncode=0)

In [2]:
import glob, json, time
from common_v8 import *
print("languages:", LANGS, "| tags:", LANG_TAG, "| vocab:", V, "| device:", DEVICE)
print("pieces per script:", {k: sum(1 for x in CLS if x == k) for k in ["hi", "gu", "pa", "latin", "common"]})
RUN = "v8_indic_continual_s0"
from huggingface_hub import HfApi
HF_REPO = HfApi().whoami()["name"] + "/cml-lang-self-checkpoints"

languages: ['hi', 'gu', 'pa'] | tags: {'hi': 4, 'gu': 5, 'pa': 6} | vocab: 32000 | device: cuda
pieces per script: {'hi': 7550, 'gu': 7768, 'pa': 7248, 'latin': 8427, 'common': 1006}


In [3]:
C = RunConfig(run_name=RUN, mode="continual", order=("hi", "gu", "pa"), n_phases=60, seed=0, time_cap_h=120.0, hf_repo=HF_REPO)
print(C)

RunConfig(run_name='v8_indic_continual_s0', mode='continual', order=('hi', 'gu', 'pa'), init_ckpt=None, n_phases=60, seed=0, max_tokens=16000, lr=0.0002, min_lr=2e-05, warmup_steps=500, cooldown_steps=2000, min_steps=2000, max_stable_steps=60000, eval_every=500, plateau_window=4, plateau_tol=0.005, weight_decay=0.1, grad_clip=1.0, masked=True, diverge_factor=1.15, max_retries=2, time_cap_h=120.0, hf_repo='s8n29/cml-lang-self-checkpoints', start_phase=None, noise_per_step=0.0, boundary_push=0.0, health_every=1, redo_every=0, redo_thresh=0.01)


In [4]:
D = load_data(train_langs=["hi", "gu", "pa"])

  hi: 993,016 train / 2,000 val
  gu: 993,016 train / 2,000 val
  pa: 993,016 train / 2,000 val
data loaded in 35s


In [ ]:
model = train_run(C, D)

 RUN v8_indic_continual_s0 | mode continual | 60 phases after phase 0 | device cuda
 phase order: init (random) → hi → gu → pa → hi → gu → pa → hi → ...
 each phase: fresh AdamW (wd 0.1 on matrices), warmup 500 → lr 0.0002 → plateau (<0.5% over 4-eval windows, after 2,000, cap 60,000) → cooldown 2,000 steps to 2e-05
 masked (script-restricted) loss & decoding: True | divergence guard: >1.15x best twice → retry at half lr (max 2)
phase 0: fresh random model (66.8M params)

 PHASE 0 DONE | trained: init | starting model, no training
--------------------------------------------------------------------------------------
   lang      val       Δ |   chrF      Δ |  chrF (unrestricted)  val (unrestr.)
   hi      9.180         |    1.6        |                  0.1          10.501
   gu      9.271         |    2.3        |                  0.4          10.565
   pa      9.148         |    1.8        |                  0.2          10.502

▶ PHASE 1/60 | training hi (993,016 pairs)


p000_init.pt:   0%|          | 0.00/267M [00:00<?, ?B/s]

  [p01 hi] step    500 | lr 2.0e-04 | train 6.784 | val 5.227 (best 5.227) | window Δ n/a | stable | 78k tok/s | phase 1m34s | run ETA: after first phase | health {'wnorm': 142.436, 'rank_enc1': 17.3, 'rank_dec1': 29.5, 'rarely_active': 0.0001}
  [p01 hi] step  1,000 | lr 2.0e-04 | train 5.357 | val 4.250 (best 4.250) | window Δ n/a | stable | 80k tok/s | phase 3m06s | run ETA: after first phase | health {'wnorm': 145.98, 'rank_enc1': 27.3, 'rank_dec1': 78.2, 'rarely_active': 0.0001}
  [p01 hi] step  1,500 | lr 2.0e-04 | train 4.737 | val 3.590 (best 3.590) | window Δ n/a | stable | 80k tok/s | phase 4m38s | run ETA: after first phase | health {'wnorm': 149.817, 'rank_enc1': 37.8, 'rank_dec1': 146.1, 'rarely_active': 0.0}


[W1103 07:36:52.995366388 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 1944059904 bytes (free: 138346496, total: 25248530432).


  [p01 hi] step  2,000 | lr 2.0e-04 | train 4.271 | val 3.127 (best 3.127) | window Δ n/a | stable | 78k tok/s | phase 6m13s | run ETA: after first phase | health {'wnorm': 153.506, 'rank_enc1': 57.5, 'rank_dec1': 218.4, 'rarely_active': 0.0}
  [p01 hi] step  2,500 | lr 2.0e-04 | train 3.967 | val 2.841 (best 2.841) | window Δ n/a | stable | 80k tok/s | phase 7m45s | run ETA: after first phase | health {'wnorm': 156.729, 'rank_enc1': 83.5, 'rank_dec1': 280.1, 'rarely_active': 0.0003}
  [p01 hi] step  3,000 | lr 2.0e-04 | train 3.781 | val 2.618 (best 2.618) | window Δ n/a | stable | 80k tok/s | phase 9m17s | run ETA: after first phase | health {'wnorm': 159.578, 'rank_enc1': 108.8, 'rank_dec1': 297.1, 'rarely_active': 0.0007}
  [p01 hi] step  3,500 | lr 2.0e-04 | train 3.588 | val 2.449 (best 2.449) | window Δ n/a | stable | 78k tok/s | phase 10m51s | run ETA: after first phase | health {'wnorm': 162.252, 'rank_enc1': 151.7, 'rank_dec1': 343.2, 'rarely_active': 0.0013}
  [p01 hi] step 

[W1103 08:31:15.091846305 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 1960837120 bytes (free: 31391744, total: 25248530432).


  [p01 hi] step 19,500 | lr 2.0e-04 | train 2.550 | val 1.544 (best 1.544) | window Δ +1.86% | stable | 78k tok/s | phase 1h00m | run ETA: after first phase | health {'wnorm': 202.672, 'rank_enc1': 527.5, 'rank_dec1': 572.0, 'rarely_active': 0.0034}
  [p01 hi] step 20,000 | lr 2.0e-04 | train 2.567 | val 1.543 (best 1.543) | window Δ +1.49% | stable | 80k tok/s | phase 1h01m | run ETA: after first phase | health {'wnorm': 203.392, 'rank_enc1': 513.4, 'rank_dec1': 557.6, 'rarely_active': 0.003}
  [p17 gu] step  1,000 | lr 2.0e-04 | train 3.083 | val 2.099 (best 2.099) | window Δ n/a | stable | 75k tok/s | phase 3m06s | run ETA ~28h26m | health {'wnorm': 230.148, 'rank_enc1': 926.4, 'rank_dec1': 751.1, 'rarely_active': 0.0272}
  [p17 gu] step  1,500 | lr 2.0e-04 | train 2.953 | val 2.078 (best 2.078) | window Δ n/a | stable | 73k tok/s | phase 4m41s | run ETA ~28h26m | health {'wnorm': 230.287, 'rank_enc1': 931.9, 'rank_dec1': 781.8, 'rarely_active': 0.0257}


In [ ]:
print_run_summary(RUN)